In [59]:
from pathlib import Path
from urllib.request import urlopen

import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq

In [6]:
# Descargar los archivos mensuales de diciembre de 2020 a 2025

ruta_actual = Path.cwd()

if ruta_actual.name == "notebooks":
    raiz_proyecto = ruta_actual.parent
else:
    raiz_proyecto = ruta_actual
    
carpeta_raw = raiz_proyecto / "data" / "raw"

carpeta_raw.mkdir(parents = True, exist_ok = True)

url_base = "https://globalmeteornetwork.org/data/traj_summary_data/monthly/"

anios_descarga = range(2020, 2026)

tamano_bloque = 1024 * 1024

for anio in anios_descarga:
    nombre_archivo = f"traj_summary_monthly_{anio}12.txt"
    
    url_archivo = f"{url_base}/{nombre_archivo}"
    
    ruta_destino = carpeta_raw / nombre_archivo
    
    ruta_temporal = ruta_destino.with_suffix(".txt.part")
    
    if ruta_destino.exists():
        print(f"{nombre_archivo}: ya existe; se omite la descarga.")
        continue
    
    print(f"\nDescargando {nombre_archivo}...")
    
    with urlopen(url_archivo, timeout = 120) as respuesta:
        tamano_total_texto = respuesta.headers.get("Content-Length")
        
        if tamano_total_texto is None:
            tamano_total = 0
        else:
            tamano_total = int(tamano_total_texto)
        
        bytes_descargados = 0
        
        with ruta_temporal.open("wb") as archivo:
            while True:
                bloque = respuesta.read(tamano_bloque)
                
                if not bloque:
                    break
                
                archivo.write(bloque)
                
                bytes_descargados += len(bloque)
                
                if tamano_total > 0:
                    porcentaje = 100 * bytes_descargados / tamano_total
                    
                    print("\r" f"{nombre_archivo}: {porcentaje:6.2f}%", end="")
    
    if tamano_total > 0 and bytes_descargados != tamano_total:
        raise IOError(f"La descarga de {nombre_archivo} quedó incompleta.")
    
    ruta_temporal.replace(ruta_destino)
    
    print("\nDescarga completada: ", ruta_destino.name)
    
registros_descargados = []

for anio in range(2019, 2026):
    nombre_archivo = f"traj_summary_monthly_{anio}12.txt"
    
    ruta_archivo = carpeta_raw / nombre_archivo
    
    registros_descargados.append(
        {
            "anio": anio,
            "archivo": nombre_archivo,
            "existe": ruta_archivo.exists(),
            "tamano_mb": (
                ruta_archivo.stat().st_size
                / 1024**2
                if ruta_archivo.exists()
                else pd.NA
            ),
        }
    )
    
resumen_descargas = pd.DataFrame(registros_descargados)

resumen_descargas["tamano_mb"] = resumen_descargas["tamano_mb"].astype("Float64").round(2)

print("\nResumen final de archivos: ")

display(resumen_descargas)

traj_summary_monthly_202012.txt: ya existe; se omite la descarga.
traj_summary_monthly_202112.txt: ya existe; se omite la descarga.
traj_summary_monthly_202212.txt: ya existe; se omite la descarga.
traj_summary_monthly_202312.txt: ya existe; se omite la descarga.
traj_summary_monthly_202412.txt: ya existe; se omite la descarga.
traj_summary_monthly_202512.txt: ya existe; se omite la descarga.

Resumen final de archivos: 


,anio,archivo,existe,tamano_mb
0,2019,traj_summary_monthly_201912.txt,True,9.85
1,2020,traj_summary_monthly_202012.txt,True,15.77
2,2021,traj_summary_monthly_202112.txt,True,29.21
3,2022,traj_summary_monthly_202212.txt,True,40.7
4,2023,traj_summary_monthly_202312.txt,True,59.96
5,2024,traj_summary_monthly_202412.txt,True,79.05
6,2025,traj_summary_monthly_202512.txt,True,104.96


In [7]:
# Definir funciones reutilizables para limpiar y reconstruir los encabezados de GMN

def limpiar_texto_encabezado(texto):
    texto_limpio = str(texto).replace("#", "")
    texto_limpio = " ".join(texto_limpio.split())
    
    return texto_limpio

def aplanar_encabezados(columnas_multinivel):
    nombres_aplanados = []
    
    for nombre_principal, nombre_secundario in columnas_multinivel:
        nombre_principal = limpiar_texto_encabezado(nombre_principal)
        nombre_secundario = limpiar_texto_encabezado(nombre_secundario)
        
        if nombre_secundario.startswith("Unnamed"):
            nombre_final = nombre_principal
        else:
            nombre_final = f"{nombre_principal} ({nombre_secundario})"
            
        nombres_aplanados.append(nombre_final)
        
    return nombres_aplanados

def renombrar_columnas_sigma(nombres_columnas):
    nombres_corregidos = nombres_columnas.copy()
    
    for indice, nombre in enumerate(nombres_corregidos):
        if nombre.startswith("+/-"):
            if indice == 0:
                raise ValueError("Se encontró una columna sigma sin una variable anterior.")
            
            variable_anterior = nombres_corregidos[indice - 1]
            nombre_variable = variable_anterior.split()[0]
            
            nombres_corregidos[indice] = f"{nombre_variable}_sigma"
    
    if len(nombres_corregidos) != len(set(nombres_corregidos)):
        nombres_repetidos = sorted({nombre for nombre in nombres_corregidos if nombres_corregidos.count(nombre) > 1})
        
        raise ValueError(f"Se generaron nombres de columnas repetidos: {nombres_repetidos}")
    
    return nombres_corregidos

In [19]:
# Convertir fechas GMN aceptano explícitamente registros con o sin fracción de segundo

def convertir_fechas_gmn(serie_fechas):
    texto_fechas = serie_fechas.astype("string").str.strip()
    
    fechas_interpretadas = pd.to_datetime(texto_fechas, format = "%Y-%m-%d %H:%M:%S.%f", errors = "coerce", utc = True)
    
    requiere_formato_segundos = fechas_interpretadas.isna() & texto_fechas.notna()
    
    fechas_solo_segundos = pd.to_datetime(texto_fechas.loc[requiere_formato_segundos], format = "%Y-%m-%d %H:%M:%S", errors = "coerce", utc = True)
    
    fechas_interpretadas.loc[requiere_formato_segundos] = fechas_solo_segundos
    
    fechas_invalidas = fechas_interpretadas.isna()
    
    if fechas_invalidas.any():
        ejemplos_invalidos = texto_fechas.loc[fechas_invalidas].head(10).tolist()
        raise ValueError(f"Se encontraron fechas que no coinciden con los formatos GMN aceptados. Ejemplos: {ejemplos_invalidos}")
    
    return fechas_interpretadas

In [20]:
# Definir una función reutilizable para cargar y preparar estructuralmente un archivo mensual de GMN

def cargar_archivo_gmn(ruta_archivo):
    ruta_archivo = Path(ruta_archivo)
    
    if not ruta_archivo.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {ruta_archivo}")
    
    df_archivo = pd.read_csv(ruta_archivo, engine = "python", sep = ";", skipinitialspace = True, skiprows = [0, 5, 6], header = [0, 1], na_values = ["nan", "None"])
    
    nombres_columnas = aplanar_encabezados(df_archivo.columns)
    nombres_columnas = renombrar_columnas_sigma(nombres_columnas)
    
    df_archivo.columns = (nombres_columnas)
    
    columnas_obligatorias = [
        "Unique trajectory (identifier)",
        "Beginning (UTC Time)",
        "IAU (code)",
        "RAgeo (deg)",
        "RAgeo_sigma",
        "DECgeo (deg)",
        "DECgeo_sigma",
        "Vgeo (km/s)",
        "Vgeo_sigma",
        "Vavg (km/s)",
        "Vavg_sigma",
        "Peak (AbsMag)",
        "Qc (deg)",
        "MedianFitErr (arcsec)",
        "Num (stat)",
        "Beg in (FOV)",
        "End in (FOV)",
        "Participating (stations)",
    ]
    
    columnas_faltantes = [columna for columna in columnas_obligatorias if columna not in df_archivo.columns]
    
    if columnas_faltantes:
        raise ValueError(f"Faltan columnas obligatorias en {ruta_archivo.name}: {columnas_faltantes}")
    
    columnas_texto = ["Unique trajectory (identifier)", "IAU (code)", "Participating (stations)"]
    
    for columna in columnas_texto:
        df_archivo[columna] = df_archivo[columna].astype("string").str.strip().replace("", pd.NA)
        
    columna_fecha = "Beginning (UTC Time)"
    
    df_archivo[columna_fecha] = convertir_fechas_gmn(df_archivo[columna_fecha])
    
    columnas_booleanas = ["Beg in (FOV)", "End in (FOV)"]
    
    for columna in columnas_booleanas:
        if not pd.api.types.is_bool_dtype(df_archivo[columna]):
            valores_booleanos = df_archivo[columna].astype("string").str.strip().str.lower()
            
            valores_invalidos = valores_booleanos.notna() & ~valores_booleanos.isin(["true", "false"])
            
            if valores_invalidos.any():
                ejemplos_invalidos = valores_booleanos.loc[valores_invalidos].drop_duplicates().tolist()
                
                raise ValueError(f"Se encontraron valores booleanos inválidos en {columna}: {ejemplos_invalidos}")
            
            if valores_booleanos.isna().any():
                raise ValueError(f"Se encontraron valores faltantes en {columna}.")
            
            df_archivo[columna] = valores_booleanos.map({"true": True, "false": False}).astype(bool)
            
    codigo_fecha = ruta_archivo.stem.split("_")[-1]
    
    if len(codigo_fecha) != 6 or not codigo_fecha.isdigit():
        raise ValueError(f"No se pudo obtener el año y el mes a partir del nombre: {ruta_archivo.name}")
    
    anio_archivo = int(codigo_fecha[:4])
    mes_archivo = int(codigo_fecha[4:])
    
    if not df_archivo[columna_fecha].dt.year.eq(anio_archivo).all():
        raise ValueError("El año de algunas fechas no coincide con el nombre del archivo.")
    
    if not df_archivo[columna_fecha].dt.month.eq(mes_archivo).all():
        raise ValueError("El mes de algunas fechas no coincide con el nombre del archivo")
    
    df_archivo["source_file"] = ruta_archivo.name
    df_archivo["source_year"] = anio_archivo
    df_archivo["source_month"] = mes_archivo
    
    return df_archivo

In [11]:
# Probar la función reutilizable con el archivo de diciembre de 2019

ruta_2019 = carpeta_raw / "traj_summary_monthly_201912.txt"

df_2019_prueba = cargar_archivo_gmn(ruta_2019)

ids_duplicados_2019 = df_2019_prueba["Unique trajectory (identifier)"].duplicated().sum()

codigos_esporadicos_2019 = df_2019_prueba["IAU (code)"].eq("...").sum()

print("Filas: ", len(df_2019_prueba))
print("Columnas: ", df_2019_prueba.shape[1])
print("Nombres de columnas únicos: ", df_2019_prueba.columns.is_unique)
print("Identificadores duplicados: ", ids_duplicados_2019)
print("Códigos esporádicos conservados: ", codigos_esporadicos_2019)
print("Fecha inicial: ", df_2019_prueba["Beginning (UTC Time)"].min())
print("Fecha final: ", df_2019_prueba["Beginning (UTC Time)"].max())
print("\nTipos de columnas clave: ")
display(
    df_2019_prueba[
        [
            "Unique trajectory (identifier)",
            "Beginning (UTC Time)",
            "IAU (code)",
            "Beg in (FOV)",
            "End in (FOV)",
            "source_file",
            "source_year",
            "source_month",
        ]
    ]
    .dtypes
    .rename("tipo")
    .to_frame()
)

Filas:  11143
Columnas:  89
Nombres de columnas únicos:  True
Identificadores duplicados:  0
Códigos esporádicos conservados:  6884
Fecha inicial:  2019-12-01 00:00:31.996809+00:00
Fecha final:  2019-12-31 23:57:18.061349+00:00

Tipos de columnas clave: 


,tipo
Unique trajectory (identifier),string
Beginning (UTC Time),"datetime64[us, UTC]"
IAU (code),string
Beg in (FOV),bool
End in (FOV),bool
source_file,str
source_year,int64
source_month,int64


In [21]:
# Definir una función que añada variables y banderas de calidad trabajando sobre una copia del DataFrame

def agregar_variables_calidad(df_entrada):
    df_calidad = df_entrada.copy(deep = True)
    
    columnas_necesarias = [
        "RAgeo (deg)",
        "RAgeo_sigma",
        "DECgeo (deg)",
        "DECgeo_sigma",
        "Vgeo_sigma",
        "Vavg_sigma",
        "Qc (deg)",
        "MedianFitErr (arcsec)",
        "Num (stat)",
        "Beg in (FOV)",
        "End in (FOV)",
    ]
    
    columnas_faltantes = [columna for columna in columnas_necesarias if columna not in df_calidad.columns]
    
    if columnas_faltantes:
        raise ValueError(f"No se pueden calcular las variables de calidad. Faltan: {columnas_faltantes}")
    
    declinacion_radianes = np.deg2rad(df_calidad["DECgeo (deg)"])
    df_calidad["RAgeo_sigma_cielo"] = df_calidad["RAgeo_sigma"] * np.cos(declinacion_radianes)
    df_calidad["sigma_radiante_aprox"] = np.hypot(df_calidad["RAgeo_sigma_cielo"], df_calidad["DECgeo_sigma"])
    df_calidad["flag_vgeo_sigma_cero"] = df_calidad["Vgeo_sigma"].eq(0)
    df_calidad["flag_vavg_sigma_cero"] = df_calidad["Vavg_sigma"].eq(0)
    
    df_calidad["cumple_qc_15"] = df_calidad["Qc (deg)"].ge(15)
    df_calidad["cumple_error_ajuste_60"] = df_calidad["MedianFitErr (arcsec)"].le(60)
    df_calidad["cumple_vavg_sigma_1"] = df_calidad["Vavg_sigma"].gt(0) & df_calidad["Vavg_sigma"].le(1)
    df_calidad["cumple_sigma_radiante_025"] = df_calidad["sigma_radiante_aprox"].le(0.25)
    
    df_calidad["calidad_radiante_publicada"] = (
        df_calidad["cumple_qc_15"] 
        & df_calidad["cumple_error_ajuste_60"] 
        & df_calidad["cumple_vavg_sigma_1"] 
        & df_calidad["cumple_sigma_radiante_025"]
    )
    
    df_calidad["fov_completo"] = (
        df_calidad["Beg in (FOV)"]
        & df_calidad["End in (FOV)"]
    )
    
    df_calidad["calidad_estandar_provisional"] = (
        df_calidad["calidad_radiante_publicada"]
        & df_calidad["fov_completo"]
    )
    
    df_calidad["calidad_estricta_provisional"] = (
        df_calidad["calidad_estandar_provisional"]
        & df_calidad["Num (stat)"].ge(4)
    )
    
    return df_calidad

In [22]:
# Validad que la función de calidad reproduzca los resultados obtenidos manualmente para 2019

df_2019_calidad = agregar_variables_calidad(df_2019_prueba)

es_geminida_2019 = df_2019_calidad["IAU (code)"].eq("GEM")

resumen_validacion_calidad = pd.DataFrame(
    {
        "comprobacion": [
            "Columnas antes de añadir calidad",
            "Columnas después de añadir calidad",
            "Vgeo_sigma igual a cero",
            "Vavg_sigma igual a cero",
            "Trayectorias estándar",
            "Gemínidas estándar",
            "Trayectorias estrictas",
            "Gemínidas estrictas",
        ],
        "resultado": [
            df_2019_prueba.shape[1],
            df_2019_calidad.shape[1],
            df_2019_calidad[
                "flag_vgeo_sigma_cero"
            ].sum(),
            df_2019_calidad[
                "flag_vavg_sigma_cero"
            ].sum(),
            df_2019_calidad[
                "calidad_estandar_provisional"
            ].sum(),
            (
                df_2019_calidad[
                    "calidad_estandar_provisional"
                ]
                & es_geminida_2019
            ).sum(),
            df_2019_calidad[
                "calidad_estricta_provisional"
            ].sum(),
            (
                df_2019_calidad[
                    "calidad_estricta_provisional"
                ]
                & es_geminida_2019
            ).sum(),
        ],
    }
)

display(resumen_validacion_calidad)

print("El DataFrame de entrada conserva", df_2019_prueba.shape[1], "columnas.")
print("El DataFrame con calidad contiene", df_2019_calidad.shape[1], "columnas.")


,comprobacion,resultado
0,Columnas antes de añadir calidad,89
1,Columnas después de añadir calidad,101
2,Vgeo_sigma igual a cero,58
3,Vavg_sigma igual a cero,0
4,Trayectorias estándar,1934
5,Gemínidas estándar,443
6,Trayectorias estrictas,457
7,Gemínidas estrictas,99


El DataFrame de entrada conserva 89 columnas.
El DataFrame con calidad contiene 101 columnas.


In [23]:
# Cargar por separado los archivos de 2019 a 2025 y generar un resumen estructural por año

columnas_esenciales_auditoria = [
    "Unique trajectory (identifier)",
    "Beginning (UTC Time)",
    "IAU (code)",
    "Sol lon (deg)",
    "RAgeo (deg)",
    "RAgeo_sigma",
    "DECgeo (deg)",
    "DECgeo_sigma",
    "Vgeo (km/s)",
    "Vgeo_sigma",
    "Vavg (km/s)",
    "Vavg_sigma",
    "Peak (AbsMag)",
    "Qc (deg)",
    "MedianFitErr (arcsec)",
    "Num (stat)",
    "Beg in (FOV)",
    "End in (FOV)",
]

datos_por_anio = {}
registros_anuales = []

for anio in range(2019, 2026):
    nombre_archivo = f"traj_summary_monthly_{anio}12.txt"
    ruta_archivo = carpeta_raw / nombre_archivo
    
    print(f"Cargando diciembre de {anio}...")
    df_anio = cargar_archivo_gmn(ruta_archivo)
    
    datos_por_anio[anio] = df_anio
    
    identificadores_duplicados = df_anio["Unique trajectory (identifier)"].duplicated().sum()
    
    filas_con_faltantes_esenciales = df_anio[columnas_esenciales_auditoria].isna().any(axis = 1).sum()
    
    geminidas = df_anio["IAU (code)"].eq("GEM").sum()
    
    memoria_mb = df_anio.memory_usage(deep = True).sum() / 1024 ** 2
    
    registros_anuales.append(
        {
            "anio": anio,
            "filas": len(df_anio),
            "columnas": df_anio.shape[1],
            "nombres_unicos": (
                df_anio.columns.is_unique
            ),
            "ids_duplicados": (
                identificadores_duplicados
            ),
            "filas_con_faltantes_esenciales": (
                filas_con_faltantes_esenciales
            ),
            "geminidas_etiquetadas": (
                geminidas
            ),
            "fecha_inicial": (
                df_anio[
                    "Beginning (UTC Time)"
                ].min()
            ),
            "fecha_final": (
                df_anio[
                    "Beginning (UTC Time)"
                ].max()
            ),
            "memoria_mb": memoria_mb,
        }
    )
    
    print(f"Completado: {len(df_anio):,} filas.")
    
resumen_estructural_anual = pd.DataFrame(registros_anuales)

resumen_estructural_anual["memoria_mb"] = resumen_estructural_anual["memoria_mb"].round(2)

print("\nResumen estructural por año: ")
display(resumen_estructural_anual)

Cargando diciembre de 2019...
Completado: 11,143 filas.
Cargando diciembre de 2020...
Completado: 17,829 filas.
Cargando diciembre de 2021...
Completado: 33,059 filas.
Cargando diciembre de 2022...
Completado: 45,785 filas.
Cargando diciembre de 2023...
Completado: 67,520 filas.
Cargando diciembre de 2024...
Completado: 89,364 filas.
Cargando diciembre de 2025...
Completado: 118,046 filas.

Resumen estructural por año: 


,anio,filas,columnas,nombres_unicos,ids_duplicados,filas_con_faltantes_esenciales,geminidas_etiquetadas,fecha_inicial,fecha_final,memoria_mb
0,2019,11143,89,True,0,0,2117,2019-12-01 00:00:31.996809+00:00,2019-12-31 23:57:18.061349+00:00,9.93
1,2020,17829,89,True,0,0,5959,2020-12-01 00:03:35.350512+00:00,2020-12-31 23:50:02.169004+00:00,15.91
2,2021,33059,89,True,0,0,9968,2021-12-01 00:00:21.427403+00:00,2021-12-31 23:59:54.288383+00:00,29.46
3,2022,45785,89,True,0,59,15800,2022-12-01 00:01:16.355611+00:00,2022-12-31 23:48:51.925169+00:00,41.05
4,2023,67520,89,True,0,87,19655,2023-12-01 00:01:51.741740+00:00,2023-12-31 23:55:18.338080+00:00,60.48
5,2024,89364,89,True,0,292,13814,2024-12-01 00:00:46.991457+00:00,2024-12-31 23:58:56.953309+00:00,79.73
6,2025,118046,89,True,0,383,37365,2025-12-01 00:00:18.917580+00:00,2025-12-31 23:58:35.114160+00:00,105.86


In [31]:
# Comparar nombres y tipos de columnas entre años y localizar los faltantes esenciales

columnas_referencia = list(datos_por_anio[2019].columns)

tipos_referencia = datos_por_anio[2019].dtypes.astype(str)

registros_esquema = []
registros_faltantes = []

for anio, df_anio in datos_por_anio.items():
    columnas_anio = list(df_anio.columns)
    
    columnas_faltantes_esquema = sorted(set(columnas_referencia) - set(columnas_anio))
    
    columnas_adicionales_esquema = sorted(set(columnas_anio) - set(columnas_referencia))
    
    columnas_comunes = [columna for columna in columnas_referencia if columna in df_anio.columns]
    
    diferencias_tipos = []
    
    for columna in columnas_comunes:
        tipo_referencia = tipos_referencia[columna]
        tipo_anio = str(df_anio[columna].dtype)
        
        if tipo_anio != tipo_referencia:
            diferencias_tipos.append(f"{columna}: {tipo_referencia} —> {tipo_anio}")
            
    registros_esquema.append(
        {
            "anio": anio,
            "mismas_columnas": (
                len(
                    columnas_faltantes_esquema
                ) == 0
                and len(
                    columnas_adicionales_esquema
                ) == 0
            ),
            "mismo_orden": (
                columnas_anio
                == columnas_referencia
            ),
            "columnas_faltantes": (
                columnas_faltantes_esquema
            ),
            "columnas_adicionales": (
                columnas_adicionales_esquema
            ),
            "cantidad_diferencias_tipos": (
                len(diferencias_tipos)
            ),
            "diferencias_tipos": (
                diferencias_tipos
            ),
        }
    )
    
    for columna in columnas_esenciales_auditoria:
        mascara_faltante = df_anio[columna].isna()
        cantidad_faltante = mascara_faltante.sum()
        
        if cantidad_faltante > 0:
            faltantes_geminidas = (mascara_faltante & df_anio["IAU (code)"].eq("GEM")).sum()
            
            registros_faltantes.append(
                {
                    "anio": anio,
                    "columna": columna,
                    "faltantes": (
                        cantidad_faltante
                    ),
                    "porcentaje_anual": round(
                        (
                            100
                            * cantidad_faltante
                            / len(df_anio)
                        ),
                        4,
                    ),
                    "faltantes_en_geminidas": (
                        faltantes_geminidas
                    ),
                }
            )
            
resumen_compatibilidad = pd.DataFrame(registros_esquema)

detalle_faltantes_esenciales = pd.DataFrame(registros_faltantes)

print("Compatibilidad estructural respecto de 2019: ")
display(resumen_compatibilidad)

print("\nFaltantes por variable y año: ")

if detalle_faltantes_esenciales.empty: 
    print("No se encontraron faltantes en las variables esenciales.")
else:
    display(detalle_faltantes_esenciales)

Compatibilidad estructural respecto de 2019: 


,anio,mismas_columnas,mismo_orden,columnas_faltantes,columnas_adicionales,cantidad_diferencias_tipos,diferencias_tipos
0,2019,True,True,[],[],0,[]
1,2020,True,True,[],[],0,[]
2,2021,True,True,[],[],0,[]
3,2022,True,True,[],[],0,[]
4,2023,True,True,[],[],0,[]
5,2024,True,True,[],[],0,[]
6,2025,True,True,[],[],0,[]



Faltantes por variable y año: 


,anio,columna,faltantes,porcentaje_anual,faltantes_en_geminidas
0,2022,MedianFitErr (arcsec),59,0.1289,17
1,2023,MedianFitErr (arcsec),87,0.1289,25
2,2024,RAgeo_sigma,292,0.3268,8
3,2024,DECgeo_sigma,292,0.3268,8
4,2024,Vgeo_sigma,292,0.3268,8
5,2024,Vavg_sigma,292,0.3268,8
6,2025,RAgeo_sigma,383,0.3244,22
7,2025,DECgeo_sigma,383,0.3244,22
8,2025,Vgeo_sigma,383,0.3244,22
9,2025,Vavg_sigma,383,0.3244,22


In [32]:
# Identificar qué variables de calidad faltan juntas y describir las trayectorias afectadas

columnas_calidad_faltantes = [
    "RAgeo_sigma",
    "DECgeo_sigma",
    "Vgeo_sigma",
    "Vavg_sigma",
    "MedianFitErr (arcsec)",
]

registros_patrones_faltantes = []

for anio, df_anio in datos_por_anio.items():
    matriz_faltantes = df_anio[columnas_calidad_faltantes].isna()
    
    patron_por_fila = matriz_faltantes.apply(lambda fila: " | ".join(fila.index[fila].tolist()), axis = 1)
    
    patrones_presentes = patron_por_fila.loc[patron_por_fila.ne("")].unique()
    
    for patron in patrones_presentes:
        mascara_patron = patron_por_fila.eq(patron)
        trayectorias_patron = df_anio.loc[mascara_patron]
        
        cantidad = len(trayectorias_patron)
        cantidad_geminidas = trayectorias_patron["IAU (code)"].eq("GEM").sum()
        
        fov_completo_patron = (
            trayectorias_patron["Beg in (FOV)"]
            & trayectorias_patron["End in (FOV)"]
        )
        
        registros_patrones_faltantes.append(
            {
                "anio": anio,
                "variables_faltantes": patron,
                "trayectorias": cantidad,
                "geminidas": (
                    cantidad_geminidas
                ),
                "estaciones_mediana": (
                    trayectorias_patron[
                        "Num (stat)"
                    ].median()
                ),
                "qc_mediana": (
                    trayectorias_patron[
                        "Qc (deg)"
                    ].median()
                ),
                "fov_completo_porcentaje": round(
                    (
                        100
                        * fov_completo_patron.mean()
                    ),
                    2,
                ),
                "ejemplo_identificador": (
                    trayectorias_patron[
                        "Unique trajectory (identifier)"
                    ].iloc[0]
                ),
            }
        )
        
resumen_patrones_faltantes = pd.DataFrame(registros_patrones_faltantes)

print("Patrones de faltantes en variables de calidad: ")

if resumen_patrones_faltantes.empty:
    print("No hay valores faltantes.")
else:
    display(resumen_patrones_faltantes)

Patrones de faltantes en variables de calidad: 


,anio,variables_faltantes,trayectorias,geminidas,estaciones_mediana,qc_mediana,fov_completo_porcentaje,ejemplo_identificador
0,2022,MedianFitErr (arcsec),59,17,3.0,60.51,100.00,20221201181356_U3QGG
1,2023,MedianFitErr (arcsec),87,25,3.0,59.40,100.00,20231201012927_EYb3P
2,2024,RAgeo_sigma | DECgeo_sigma | Vgeo_sigma | Vavg...,292,8,2.0,16.53,99.66,20241201011201_FbsEY
3,2025,RAgeo_sigma | DECgeo_sigma | Vgeo_sigma | Vavg...,383,22,2.0,12.81,100.00,20251201023655_9tn3K


In [50]:
# Preparar las variables de calidad y marcar explícitamente las trayectorias con información de calidad incompleta

def preparar_calidad_gmn(df_entrada):
    df_calidad = agregar_variables_calidad(df_entrada)
    
    columnas_requeridas_calidad = [
        "RAgeo_sigma",
        "DECgeo_sigma",
        "Vgeo_sigma",
        "Vavg_sigma",
        "MedianFitErr (arcsec)",
    ]
    
    df_calidad["flag_calidad_incompleta"] = df_calidad[columnas_requeridas_calidad].isna().any(axis = 1)
    
    df_calidad["calidad_radiante_publicada"] = df_calidad["calidad_radiante_publicada"] & ~df_calidad["flag_calidad_incompleta"]
    df_calidad["calidad_estandar_provisional"] = (
        df_calidad["calidad_radiante_publicada"] 
        & df_calidad["fov_completo"]
    )
    df_calidad["calidad_estricta_provisional"] = (
        df_calidad["calidad_estandar_provisional"]
        & df_calidad["Num (stat)"].ge(4)
    )
    
    return df_calidad

In [51]:
# Comrpobar que las trayectorias con calidad incompleta se conserven pero no entren en las muestras filtradas

df_2022_calidad_prueba = preparar_calidad_gmn(datos_por_anio[2022])
df_2024_calidad_prueba = preparar_calidad_gmn(datos_por_anio[2024])

resumen_control_faltantes = pd.DataFrame(
    {
        "anio": [
            2022,
            2024,
        ],
        "calidad_incompleta": [
            df_2022_calidad_prueba[
                "flag_calidad_incompleta"
            ].sum(),
            df_2024_calidad_prueba[
                "flag_calidad_incompleta"
            ].sum(),
        ],
        "geminidas_con_calidad_incompleta": [
            (
                df_2022_calidad_prueba[
                    "flag_calidad_incompleta"
                ]
                & df_2022_calidad_prueba[
                    "IAU (code)"
                ].eq("GEM")
            ).sum(),
            (
                df_2024_calidad_prueba[
                    "flag_calidad_incompleta"
                ]
                & df_2024_calidad_prueba[
                    "IAU (code)"
                ].eq("GEM")
            ).sum(),
        ],
        "incompletas_en_muestra_estandar": [
            (
                df_2022_calidad_prueba[
                    "flag_calidad_incompleta"
                ]
                & df_2022_calidad_prueba[
                    "calidad_estandar_provisional"
                ]
            ).sum(),
            (
                df_2024_calidad_prueba[
                    "flag_calidad_incompleta"
                ]
                & df_2024_calidad_prueba[
                    "calidad_estandar_provisional"
                ]
            ).sum(),
        ],
        "filas_conservadas": [
            len(
                df_2022_calidad_prueba
            ),
            len(
                df_2024_calidad_prueba
            ),
        ],
    }
)

display(resumen_control_faltantes)

,anio,calidad_incompleta,geminidas_con_calidad_incompleta,incompletas_en_muestra_estandar,filas_conservadas
0,2022,59,17,0,45785
1,2024,292,8,0,89364


In [52]:
# Añadir variables de calidad a cada año, concatenar los siete DataFrames y auditar la integración

dataframes_preparados = []

for anio in range(2019, 2026):
    print(f"Preparando calidad para {anio}")
    
    df_anio_preparado = preparar_calidad_gmn(datos_por_anio[anio])
    
    dataframes_preparados.append(df_anio_preparado)
    
    print(f"Completado: {len(df_anio_preparado):,} filas.")
    
df_integrado = pd.concat(dataframes_preparados, ignore_index = True, sort = False)

filas_esperadas = sum(len(df_anio) for df_anio in datos_por_anio.values())

if len(df_integrado) != filas_esperadas:
    raise ValueError("El número de filas depsués de concatenar no coincide con la suma de los archivos individuales.")

if not df_integrado.columns.is_unique:
    raise ValueError("La base integrada contiene nombres de columnas repetidos.")

ids_duplicados_globales = df_integrado["Unique trajectory (identifier)"].duplicated().sum()

registros_integracion = []

for anio, df_anio in df_integrado.groupby("source_year", sort = True):
    es_geminida = df_anio["IAU (code)"].eq("GEM")
    cantidad_geminidas = es_geminida.sum()
    
    estandar = df_anio["calidad_estandar_provisional"]
    estricta = df_anio["calidad_estricta_provisional"]
    
    geminidas_estandar = (es_geminida & estandar).sum()
    geminidas_estrictas = (es_geminida & estricta).sum()
    
    filas_originales = len(datos_por_anio[anio])
    
    registros_integracion.append(
        {
            "anio": anio,
            "filas_originales": (
                filas_originales
            ),
            "filas_integradas": (
                len(df_anio)
            ),
            "filas_coinciden": (
                filas_originales
                == len(df_anio)
            ),
            "calidad_incompleta": (
                df_anio[
                    "flag_calidad_incompleta"
                ].sum()
            ),
            "geminidas_totales": (
                cantidad_geminidas
            ),
            "geminidas_estandar": (
                geminidas_estandar
            ),
            "retencion_estandar_gem_porcentaje": (
                round(
                    (
                        100
                        * geminidas_estandar
                        / cantidad_geminidas
                    ),
                    2,
                )
                if cantidad_geminidas > 0
                else pd.NA
            ),
            "geminidas_estrictas": (
                geminidas_estrictas
            ),
            "retencion_estricta_gem_porcentaje": (
                round(
                    (
                        100
                        * geminidas_estrictas
                        / cantidad_geminidas
                    ),
                    2,
                )
                if cantidad_geminidas > 0
                else pd.NA
            ),
        }
    )
    
resumen_integracion = pd.DataFrame(registros_integracion)

memoria_integrada_mb = df_integrado.memory_usage(deep = True).sum() / 1024 ** 2

print("\nFilas esperadas: ", filas_esperadas)
print("Filas integradas: ", len(df_integrado))
print("Columnas integradas: ", df_integrado.shape[1])
print("Nombres de columnas únicos: ", df_integrado.columns.is_unique)
print("Identificadores duplicados globales: ", ids_duplicados_globales)
print("Memoria de la base integrada: ", round(memoria_integrada_mb, 2), "MB")
print("\nResumen de integracion por año: ")

display(resumen_integracion)

Preparando calidad para 2019
Completado: 11,143 filas.
Preparando calidad para 2020
Completado: 17,829 filas.
Preparando calidad para 2021
Completado: 33,059 filas.
Preparando calidad para 2022
Completado: 45,785 filas.
Preparando calidad para 2023
Completado: 67,520 filas.
Preparando calidad para 2024
Completado: 89,364 filas.
Preparando calidad para 2025
Completado: 118,046 filas.

Filas esperadas:  382746
Filas integradas:  382746
Columnas integradas:  102
Nombres de columnas únicos:  True
Identificadores duplicados globales:  0
Memoria de la base integrada:  352.27 MB

Resumen de integracion por año: 


,anio,filas_originales,filas_integradas,filas_coinciden,calidad_incompleta,geminidas_totales,geminidas_estandar,retencion_estandar_gem_porcentaje,geminidas_estrictas,retencion_estricta_gem_porcentaje
0,2019,11143,11143,True,0,2117,443,20.93,99,4.68
1,2020,17829,17829,True,0,5959,1051,17.64,191,3.21
2,2021,33059,33059,True,0,9968,2023,20.29,322,3.23
3,2022,45785,45785,True,59,15800,3769,23.85,1776,11.24
4,2023,67520,67520,True,87,19655,3538,18.00,1335,6.79
5,2024,89364,89364,True,292,13814,2622,18.98,621,4.50
6,2025,118046,118046,True,383,37365,6312,16.89,2924,7.83


In [58]:
# Guardar la base integrada como Parquet y verificar que el archivo conserve todas las filas y columnas

carpeta_interim = raiz_proyecto / "data" / "interim"
carpeta_tablas = raiz_proyecto / "reports" / "tables"

carpeta_interim.mkdir(parents = True, exist_ok = True)
carpeta_tablas.mkdir(parents = True, exist_ok = True)

ruta_parquet = carpeta_interim / "gmn_diciembre_2019_2025.parquet"
ruta_parquet_temporal = carpeta_interim / "gmn_diciembre_2019_2025.parquet.part"

ruta_resumen_csv = carpeta_tablas / "resumen_integracion_2019_2025.csv"

print("Guardando la base integrada...")

df_integrado.to_parquet(ruta_parquet_temporal, engine = "pyarrow", compression = "zstd", index = False)

ruta_parquet_temporal.replace(ruta_parquet)

resumen_integracion.to_csv(ruta_resumen_csv, index = False, encoding = "utf-8")

archivo_parquet = pq.ParquetFile(ruta_parquet)

filas_guardadas = archivo_parquet.metadata.num_rows
columnas_guardadas = archivo_parquet.metadata.num_columns
tamano_parquet_mb = ruta_parquet.stat().st_size / 1024 ** 2

if filas_guardadas != len(df_integrado):
    raise ValueError("El archivo Parquet no contiene el número esperado de filas.")

if columnas_guardadas != df_integrado.shape[1]:
    raise ValueError["El archivo Parquet no contiene el número esperado de columnas."]

print("Guardado completado correctamente.")
print("\nRuta del Parquet: ", ruta_parquet)
print("Filas guardadas: ", filas_guardadas)
print("Columnas guardadas: ", columnas_guardadas)
print("Tamaño del Parquet: ", round(tamano_parquet_mb, 2), "MB")
print("\nRuta del resumen CSV: ", ruta_resumen_csv)
print("Filas del resumen: ", len(resumen_integracion))

Guardando la base integrada...
Guardado completado correctamente.

Ruta del Parquet:  /Users/alexn/Documents/Uni_Itesm/Tercer semestre/F1009/Investigacion/data/interim/gmn_diciembre_2019_2025.parquet
Filas guardadas:  382746
Columnas guardadas:  102
Tamaño del Parquet:  142.17 MB

Ruta del resumen CSV:  /Users/alexn/Documents/Uni_Itesm/Tercer semestre/F1009/Investigacion/reports/tables/resumen_integracion_2019_2025.csv
Filas del resumen:  7
